In [2]:
%run ./dataManipulationFunctions.ipynb

In [3]:
columns = ['clipID', 'Extraversion', 'Agreeableness', 'Conscientiousness', 'Neuroticism', 'Openness', 'Gender']
listOfGenders = np.array(metadata)[:,2]
unique_values, counts = np.unique(listOfGenders, return_counts=True)

In [4]:
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder

2023-12-11 14:11:43.028370: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [5]:
labelencoder=LabelEncoder()
y=to_categorical(labelencoder.fit_transform(listOfGenders))[:,0]
# No of men vs no of women
print(pd.DataFrame(y).value_counts())
# % of men vs % of women
print(pd.DataFrame(y).value_counts(normalize=True) * 100)

#MALE - 0, FEMALE - 1

0.0    503
1.0    137
Name: count, dtype: int64
0.0    78.59375
1.0    21.40625
Name: proportion, dtype: float64


In [6]:
from sklearn.model_selection import train_test_split
# Getting median scores per clip as the feature values
X = medianScoresPerClip()[:,1:]


In [14]:
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

In [15]:
# from sklearn.model_selection import RepeatedStratifiedKFold
# from sklearn.neighbors import KNeighborsClassifier
# #n_splits - k no of folds
# # n_repeats, number of times cross-validation is repeated

# models=[LogisticRegression(solver='liblinear',class_weight='balanced'), KNeighborsClassifier(n_neighbors=2, weights='distance'), DecisionTreeClassifier(class_weight={0:3.6715328467153285,1:1})]
# for model in models:
#     cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=0)
#     scores = cross_val_score(model, X_trainVal, y_trainVal, scoring="f1", cv=cv)

#     print(model,(scores))


Splitting Dataset into two, a set for training and validation data and one for testing data.

K fold validation is used as a way to create a function which takes an algoritm and using k folds, calculates its f1 score.  
    K fold splits the train set into k train/validation sets which are used to determine hyperparameters etc.

In [55]:
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix, precision_score, recall_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=1)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

logisticReg = LogisticRegression(solver='liblinear',class_weight='balanced')
Kneightbors = KNeighborsClassifier(n_neighbors=5, weights='distance')
randomForest = RandomForestClassifier(n_estimators=200, class_weight='balanced')
decisionTree=DecisionTreeClassifier(class_weight={0:3.6715328467153285,1:1})
SVM = SVC(kernel='poly', C=6, degree=5)

kFold = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

def calculateResults(model, X, y):
    f1Scores=[]
    confusionMatrices=[]
    confusionMatrixTotal=[[0,0],[0,0]]
    for trainIndex, valIndex in kFold.split(X, y):
        X_tr, X_te = X[trainIndex], X[valIndex]
        y_tr, y_te = y[trainIndex], y[valIndex]
        model.fit(X_tr, y_tr)
        prediction = model.predict(X_te)
        f1Scores.append(f1_score(y_te, prediction))
        confusionMatrices.append(confusion_matrix(y_te, prediction))
        confusionMatrixTotal += np.array(confusion_matrix(y_te, prediction))
        
    # display(pd.DataFrame({'f1 score': f1Scores, 'confusion matrix': confusionMatrices}))
    result = pd.DataFrame({'Average f1 score': np.mean(f1Scores), 'Total confusion matrix': ([confusionMatrixTotal])})
    return (result)

algorithmName = ["Logistic Regression", "Random Forest", "Decision Tree", "SVM", "K neighbours"]
algorithms = [logisticReg, randomForest, decisionTree, SVM, Kneightbors]
df = pd.DataFrame()
for i in range (len(algorithms)):
    df = pd.concat([df,calculateResults(algorithms[i],X_train, y_train)])
df.index = algorithmName

display(df)


,Average f1 score,Total confusion matrix
Logistic Regression,0.477645,"[[272, 130], [35, 75]]"
Random Forest,0.315439,"[[314, 88], [73, 37]]"
Decision Tree,0.249938,"[[358, 44], [88, 22]]"
SVM,0.261507,"[[369, 33], [88, 22]]"
K neighbours,0.250524,"[[368, 34], [89, 21]]"


In [44]:
def calculateFinal(model):
    model.fit(X_train, y_train)
    prediction = model.predict(X_test)
    f1Score = (f1_score(y_test, prediction))
    confusionMatrix =(confusion_matrix(y_test, prediction))
    return f1Score, confusionMatrix

f1Scores = [calculateResults(logisticReg, X_test, y_test)['Average f1 score'],calculateResults(randomForest, X_test, y_test)['Average f1 score'], calculateResults(decisionTree, X_test, y_test)['Average f1 score'], calculateResults(SVM, X_test, y_test)['Average f1 score'], calculateResults(Kneightbors, X_test, y_test)['Average f1 score'] ]
confusionMatrices = [calculateResults(logisticReg, X_test, y_test)['Total confusion matrix'],calculateResults(randomForest, X_test, y_test)['Total confusion matrix'], calculateResults(decisionTree, X_test, y_test)['Total confusion matrix'], calculateResults(SVM, X_test, y_test)['Total confusion matrix'], calculateResults(Kneightbors, X_test, y_test)['Total confusion matrix'] ]

confusionMatricesF = [calculateFinal(logisticReg)[1],calculateFinal(randomForest)[1], calculateFinal(decisionTree)[1], calculateFinal(SVM)[1], calculateFinal(Kneightbors)[1] ]

algorithms = [logisticReg, randomForest, decisionTree, SVM, Kneightbors]

testSetResults = pd.DataFrame()
for i in range (len(algorithms)):
    testSetResults = pd.concat([testSetResults, calculateResults(algorithms[i], X_test, y_test)])
    # print(calculateFinal(algorithms[i])[1], calculateResults(algorithms[i], X_test, y_test)['Total confusion matrix'])
testSetResults.index = algorithmName
# display(testSetResults)


In [56]:
from scipy import stats

binomialTestResults = pd.DataFrame()
confusionMatrices = [calculateResults(logisticReg, X_test, y_test)['Total confusion matrix'],calculateResults(randomForest, X_test, y_test)['Total confusion matrix'], calculateResults(decisionTree, X_test, y_test)['Total confusion matrix'], calculateResults(SVM, X_test, y_test)['Total confusion matrix'], calculateResults(Kneightbors, X_test, y_test)['Total confusion matrix'] ]
# confusionMatricesF = [calculateFinal(logisticReg)[1],calculateFinal(randomForest)[1], calculateFinal(decisionTree)[1], calculateFinal(SVM)[1], calculateFinal(Kneightbors)[1] ]


for i in range(5):
    algorithm = algorithms[i]
    testSetSize = y_test.size
    # confusionMatrix = getTestResults()['Total confusion matrix'][algorithm]
    # print(confusionMatrix)
    trueNeg = confusionMatrices[i][0][0][0]  #correct male (got 0 is 0)
    falsePos = confusionMatrices[i][0][0][1] #wrong male    (got 1 is 0)
    falseNeg = confusionMatrices[i][0][1][0] #wrong female  (got 0 is 1)
    truePos = confusionMatrices[i][0][1][1]  #correct female (got 1 is 1)
    
    totalMale,totalFemale = pd.DataFrame(y).value_counts()
    wholeSetFemaleProb = totalFemale/y.size
    wholeSetMaleProb = totalMale/y.size

    noOfSuccesses = trueNeg + truePos
    p = ((wholeSetFemaleProb**2) + (wholeSetMaleProb**2))
    result = stats.binomtest(noOfSuccesses, n=testSetSize, p=p)
    values= pd.DataFrame([[f1Scores[i][0], result.pvalue]])
    binomialTestResults = pd.concat([binomialTestResults,values])

binomialTestResults.rename(columns={0: 'f1 value for test set', 1: 'Binomial test p value'}, inplace=True)
binomialTestResults.index = algorithmName
display(binomialTestResults)

,f1 value for test set,Binomial test p value
Logistic Regression,0.439086,0.925518
Random Forest,0.227772,0.039552
Decision Tree,0.186667,0.399863
SVM,0.236032,0.024575
K neighbours,0.175758,0.092834
